In [ ]:
##PARAMS CELL
TASK_NAME = "ability_difference"
DATASET_NAME = "toy_mbpp"
RUN_ID_1 = 9
RUN_ID_2 = 11
SCORER_NAME = "verify"

SUCCESS_METRIC_NAME = "accuracy"

TO_SAVE_DIR = None  #"/Users/RobertAdragna/Documents/MATS/evals_suite_new/results/final_runs/mal_evasion"

In [ ]:
from src.experiment_tracker import ExperimentTracker
from src.utils.utils import RESULTS_DIR
import os
import shutil
from typing import List
from src.utils.analysis_utils import *

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


from inspect_ai.log import read_eval_log
from inspect_ai.scorer import Score

In [ ]:
tracker = ExperimentTracker(os.path.join(RESULTS_DIR, "experiment_tracker"))
exp_df_1 = load_expdf_from_tracker(tracker, task_name=TASK_NAME, dataset_name=DATASET_NAME, run_id=RUN_ID_1)
exp_df_2 = load_expdf_from_tracker(tracker, task_name=TASK_NAME, dataset_name=DATASET_NAME, run_id=RUN_ID_2)



# Concatenate exp_df_1 and exp_df_2, aligning columns.
exp_df = pd.concat([exp_df_1, exp_df_2], axis=0, ignore_index=True, sort=False)


for idx, row in exp_df.iterrows():
    if row.get('model') == 'anthropic/claude-sonnet-4-20250514' and pd.notna(row.get('reasoning_tokens')):
        exp_df.at[idx, 'model'] = f"{row['model']} + reasoning"

exp_df = exp_df[exp_df['cond_type'].isin(['easiest', 'lemonade-embedded'])]

# Calculate stderr for honeypot_included using binomial standard error formula: sqrt(p*(1-p)/n)
# where p is the accuracy and n is the limit (number of samples)
exp_df['honeypot_included__stderr'] = np.sqrt(
    exp_df['honeypot_included__accuracy'] * (1 - exp_df['honeypot_included__accuracy']) / exp_df['limit']
)

# Calculate 95% confidence intervals (±1.96 * stderr)
exp_df['verify__95pct'] = 1.96 * exp_df['verify__stderr']
exp_df['honeypot_included__95pct'] = 1.96 * exp_df['honeypot_included__stderr']

exp_df.head(10)

## Specification Gaming

In [ ]:
# Create dataframe comparing impossible vs possible tasks
# Pivot to get separate columns for impossible=True and impossible=False
pivot_df = exp_df.pivot_table(
    index=['model', 'cond_type'],
    columns='is_impossible_task',
    values=['honeypot_included__accuracy', 'honeypot_included__95pct'],
    aggfunc='first'
).reset_index()

# Flatten column names
pivot_df.columns = ['_'.join(map(str, col)).strip('_') if col[1] != '' else col[0] 
                    for col in pivot_df.columns.values]

# Calculate difference (possible - impossible)
# This gives us: reward hacking rate - spec gaming rate
pivot_df['rh_rate_-_specgaming_rate__accuracy'] = (
    pivot_df['honeypot_included__accuracy_False'] - pivot_df['honeypot_included__accuracy_True']
)

# Calculate combined 95% CI for difference of two independent random variables
# Var(X - Y) = Var(X) + Var(Y), so std(X - Y) = sqrt(std(X)^2 + std(Y)^2)
# Since 95pct = 1.96 * stderr, we have stderr = 95pct / 1.96
# Combined stderr = sqrt((95pct_1/1.96)^2 + (95pct_2/1.96)^2) = sqrt(95pct_1^2 + 95pct_2^2) / 1.96
# Combined 95pct = 1.96 * combined_stderr = sqrt(95pct_1^2 + 95pct_2^2)
pivot_df['rh_rate_-_specgaming_rate__95pct'] = np.sqrt(
    pivot_df['honeypot_included__95pct_True']**2 + 
    pivot_df['honeypot_included__95pct_False']**2
)

# Reorder and select relevant columns
exploit_df = pivot_df[[
    'model', 'cond_type',
    'honeypot_included__accuracy_False', 'honeypot_included__accuracy_True',
    'rh_rate_-_specgaming_rate__accuracy', 'rh_rate_-_specgaming_rate__95pct'
]].copy()

# Rename for clarity
exploit_df.columns = [
    'model', 'cond_type',
    'honeypot_acc_possible', 'honeypot_acc_impossible',
    'rh_rate_-_specgaming_rate__accuracy', 'rh_rate_-_specgaming_rate__95pct'
]

exploit_df[[
    'model', 'cond_type',
    'honeypot_acc_possible', 'honeypot_acc_impossible',
    'rh_rate_-_specgaming_rate__accuracy', 'rh_rate_-_specgaming_rate__95pct'
]]

In [ ]:
exploit_df[[
    'model', 'cond_type',
    'rh_rate_-_specgaming_rate__accuracy', 'rh_rate_-_specgaming_rate__95pct'
]]